Before running this notebook, please run:

```bash
pixi run s2gos-server run --service=s2gos_example_app.app:service
```

This will start the server on http://127.0.0.1:8008 with the correct service path.

In [1]:
from pathlib import Path
from s2gos_generator.core.config import (
    create_scene_config,
    MolecularAtmosphereConfig,
    ThermophysicalConfig,
    ParticleLayerConfig,
    AbsorptionDatabase,
    AerosolDataset,
    ExponentialDistribution,
)
from s2gos_simulator.config import (
    SimulationConfig,
    UAVSensor,
    DirectionalIllumination,
    LookAtViewing,
    SpectralResponse,
    UAVInstrumentType,
)

from s2gos_client import Client
from s2gos_common.models import ProcessRequest

2025-07-16 17:47:31,988 - INFO - Registering profile afgl_1986-tropical
2025-07-16 17:47:31,989 - INFO - Registering profile afgl_1986-midlatitude_summer
2025-07-16 17:47:31,989 - INFO - Registering profile afgl_1986-midlatitude_winter
2025-07-16 17:47:31,990 - INFO - Registering profile afgl_1986-subarctic_summer
2025-07-16 17:47:31,990 - INFO - Registering profile afgl_1986-subarctic_winter
2025-07-16 17:47:31,990 - INFO - Registering profile afgl_1986-us_standard
2025-07-16 17:47:31,991 - INFO - Registering profile mipas_2007-midlatitude_day
2025-07-16 17:47:31,991 - INFO - Registering profile mipas_2007-midlatitude_night
2025-07-16 17:47:31,992 - INFO - Registering profile mipas_2007-polar_summer
2025-07-16 17:47:31,992 - INFO - Registering profile mipas_2007-polar_winter
2025-07-16 17:47:31,992 - INFO - Registering profile mipas_2007-tropical
2025-07-16 17:47:31,992 - INFO - Registering profile ussa_1976


[mitsuba] Warning: Couldn't import the ipywidgets package. Installing this package is required for the system to properly log messages and print in Jupyter notebooks!


/home/gonzalezm/s2gos-example-app/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-16 17:47:32,319 - INFO - Detected pandas and successfully loaded Hamilton extensions.
2025-07-16 17:47:32,365 - INFO - Detected dask and successfully loaded Hamilton extensions.
2025-07-16 17:47:32,365 - INFO - Detected geopandas and successfully loaded Hamilton extensions.
2025-07-16 17:47:32,367 - INFO - Cannot import pandera from pandera_validators. Run pip install sf-hamilton[pandera] if needed.


In [2]:
# Initialize the client (server runs on port 8008 by default)
client = Client(server_url="http://127.0.0.1:8008")

SERVER_DATA_ROOT = Path("./")
SERVER_OUTPUT_ROOT = Path("./")
SERVER_GENERATOR_REPO_ROOT = Path("./")

In [ ]:
config = create_scene_config(
    scene_name="notebook_test_scene",
    center_lat=-23.6002,
    center_lon=15.11956,
    aoi_size_km=10.0,
    output_dir=Path("./notebook_output").resolve(),  # Convert to absolute path
    target_resolution_m=30.0,
    description="A scene generated from the s2gos-client notebook using working examples approach",
)

config.enable_buffer_system(
    buffer_size_km=60.0,
    buffer_resolution_m=100.0,
    background_elevation=0.0,
    background_resolution_m=200.0,
)

molecular_config = MolecularAtmosphereConfig(
    thermoprops=ThermophysicalConfig(
        identifier="afgl_1986-us_standard",
    ),
    absorption_database=AbsorptionDatabase.GECKO,
    has_absorption=True,
    has_scattering=True,
)

hazy_layer = ParticleLayerConfig(
    aerosol_dataset=AerosolDataset.SIXSV_CONTINENTAL,
    optical_thickness=0.3,
    altitude_bottom=0.0,
    altitude_top=1000.0,
    distribution=ExponentialDistribution(rate=5.0),
    reference_wavelength=550.0,
    has_absorption=True,
)

config.set_atmosphere_heterogeneous(
    molecular_config=molecular_config, particle_layers=[hazy_layer]
)

print("Configuration created")
print(f"Scene: {config.scene_name}")
print(f"Location: {config.location.center_lat:.4f}°, {config.location.center_lon:.4f}°")
print(f"AOI: {config.location.aoi_size_km} km²")
print(f"Resolution: {config.processing.target_resolution_m} m")
print(f"Output directory (absolute): {config.output_dir}")
print(f"Scene output directory (absolute): {config.scene_output_dir}")

errors = config.validate_configuration()
if errors:
    print(f"Configuration errors: {errors}")
else:
    print("Configuration validation passed")

Configuration created
Scene: notebook_test_scene
Location: -23.6002°, 15.1196°
AOI: 10.0 km²
Resolution: 30.0 m
Output directory (absolute): /home/gonzalezm/s2gos-example-app/notebook/notebook_output
Scene output directory (absolute): /home/gonzalezm/s2gos-example-app/notebook/notebook_output/notebook_test_scene
Configuration validation passed


In [4]:
# Execute scene generation process
job = client.execute_process(
    process_id="generate_scene",
    request=ProcessRequest(
        inputs={
            "config": config.model_dump(mode="json")
        }
    ),
)

print("Scene generation job submitted:")
print(f"Job ID: {job.jobID}")
print(f"Status: {job.status}")
print(f"Progress: {job.progress}%")
print(job)

2025-07-16 17:47:32,575 - INFO - HTTP Request: POST http://127.0.0.1:8008/processes/generate_scene/execution "HTTP/1.1 201 Created"


Scene generation job submitted:
Job ID: job_0
Status: JobStatus.running
Progress: None%
processID='generate_scene' type=<JobType.process: 'process'> jobID='job_0' status=<JobStatus.running: 'running'> message=None created=datetime.datetime(2025, 7, 16, 17, 47, 32, 574742) started=datetime.datetime(2025, 7, 16, 17, 47, 32, 574907) finished=None updated=None progress=None links=None traceback=None


In [5]:
# Check all jobs status
jobs = client.get_jobs()
print("All jobs:")
for job in jobs.jobs:
    print(f"  Job ID: {job.jobID}")
    print(f"  Process: {job.processID}")
    print(f"  Status: {job.status}")
    print(f"  Progress: {job.progress}%")
    print(f"  Created: {job.created}")
    print("  ---")
    
jobs

2025-07-16 17:47:32,586 - INFO - HTTP Request: GET http://127.0.0.1:8008/jobs "HTTP/1.1 200 OK"


All jobs:
  Job ID: job_0
  Process: generate_scene
  Status: JobStatus.running
  Progress: None%
  Created: 2025-07-16 17:47:32.574742
  ---


JobList(jobs=[JobInfo(processID='generate_scene', type=<JobType.process: 'process'>, jobID='job_0', status=<JobStatus.running: 'running'>, message=None, created=datetime.datetime(2025, 7, 16, 17, 47, 32, 574742), started=datetime.datetime(2025, 7, 16, 17, 47, 32, 574907), finished=None, updated=None, progress=None, links=None, traceback=None)], links=[Link(href='http://127.0.0.1:8008/jobs', rel='self', type='application/json', hreflang='en', title='get_jobs')])

In [6]:
# Clean up completed jobs (optional)
for job in client.get_jobs().jobs:
    if job.status == "successful" or job.status == "failed":
        client.dismiss_job(job.jobID)
        print(f"Dismissed job {job.jobID} ({job.status})")

print("Job cleanup completed")

2025-07-16 17:47:32,604 - INFO - HTTP Request: GET http://127.0.0.1:8008/jobs "HTTP/1.1 200 OK"


Job cleanup completed


In [ ]:
sensors = [
    UAVSensor(
        id="uav_rgb_camera",
        instrument=UAVInstrumentType.PERSPECTIVE_CAMERA,
        viewing=LookAtViewing(origin=[0, 0, 55000], target=[0, 0, 0], up=[0, 1, 0]),
        srf=SpectralResponse(type="delta", wavelengths=[440.0, 550.0, 660.0]),
        fov=70.0,
        resolution=[1024, 1024],
        samples_per_pixel=128,
    ),
]

simulation_config = SimulationConfig(
    name="notebook_simulation",
    description="Simulation from notebook using working examples approach",
    illumination=DirectionalIllumination(zenith=30.0, azimuth=180.0),
    sensors=sensors,
    backend_hints={"eradiate": {"mode": "mono"}},
)

print("Simulation configuration created:")
print(f"  Name: {simulation_config.name}")
print(f"  Sensors: {len(simulation_config.sensors)}")
for i, sensor in enumerate(simulation_config.sensors):
    print(f"    {i + 1}. {sensor.id}")
print(f"  Illumination: zenith={simulation_config.illumination.zenith}°, azimuth={simulation_config.illumination.azimuth}°")

Simulation configuration created:
  Name: notebook_simulation
  Sensors: 1
    1. uav_rgb_camera
  Illumination: zenith=30.0°, azimuth=180.0°


In [ ]:
scene_file_path = str(config.scene_output_dir / f"{config.scene_name}.yml")
simulation_output_path = str(config.output_dir / "simulation_results")

print(f"Scene file path: {scene_file_path}")
print(f"Simulation output path: {simulation_output_path}")

simulation_job = client.execute_process(
    process_id="simulate_observation",
    request=ProcessRequest(
        inputs={
            "scene_file_path": scene_file_path,
            "simulation_config": simulation_config.model_dump(mode="json"),
            "output_path": simulation_output_path  # Separate output directory for results
        }
    ),
)

print("Simulation job submitted:")
print(f"Job ID: {simulation_job.jobID}")
print(f"Status: {simulation_job.status}")
print(f"Progress: {simulation_job.progress}%")
print(simulation_job)

2025-07-16 17:03:49,061 - INFO - HTTP Request: POST http://127.0.0.1:8008/processes/simulate_observation/execution "HTTP/1.1 201 Created"


Scene file path: /home/gonzalezm/s2gos-example-app/notebook/notebook_output/notebook_test_scene/notebook_test_scene.yml
Simulation output path: /home/gonzalezm/s2gos-example-app/notebook/notebook_output/simulation_results
Simulation job submitted:
Job ID: job_1
Status: JobStatus.accepted
Progress: None%
processID='simulate_observation' type=<JobType.process: 'process'> jobID='job_1' status=<JobStatus.accepted: 'accepted'> message=None created=datetime.datetime(2025, 7, 16, 17, 3, 49, 60565) started=None finished=None updated=None progress=None links=None traceback=None


In [10]:
# Check all jobs status
jobs = client.get_jobs()
print("All jobs:")
for job in jobs.jobs:
    print(f"  Job ID: {job.jobID}")
    print(f"  Process: {job.processID}")
    print(f"  Status: {job.status}")
    print(f"  Progress: {job.progress}%")
    print(f"  Created: {job.created}")
    print("  ---")
    
jobs

ConnectError: [Errno 111] Connection refused